# RoBoat Run C (V7 Flagship) — Control Arm Stage-2 Completion

This notebook executes **Stage 2 (Flagship Convergence & Native-Resolution Fine-Tuning)** of the COCO-initialized control arm on a Kaggle Tesla T4 GPU.

- Resumes from Epoch 44 `last.pt` (dptel22/roboat-run-c-v4-output)
- Trains to 100 epochs with `close_mosaic=15`, `scale=0.25`, `mosaic=0.80`, `batch=32`, `workers=4`
- Two-hook session guard (`on_model_save` backup + `on_fit_epoch_end` timer)
- Trajectory stitching (`results_full_trajectory.csv`)
- Dual-Protocol Stratified Evaluation: Protocol A (Standard iou=0.60) and Protocol B (Multi-scale TTA iou=0.55)


In [ ]:
%pip install -q ultralytics==8.4.165 albumentations==2.0.8 PyYAML==6.0.3


In [ ]:
# The notebook embeds the matching CLI script, so no separate Kaggle code input is needed.
import base64
import os
import subprocess
import sys
from pathlib import Path

# Print mounted datasets for verification
input_dir = Path('/kaggle/input')
if input_dir.is_dir():
    print('Mounted in /kaggle/input:', [p.name for p in input_dir.iterdir()])

# Auto-discover DATA_ROOT
DATA_ROOT = Path('/kaggle/input/roboat-run-c')
if not (DATA_ROOT / 'yamls').is_dir() and input_dir.is_dir():
    for p in input_dir.rglob('combined_c2.yaml'):
        DATA_ROOT = p.parent.parent
        print(f'[data-root] Auto-discovered DATA_ROOT: {DATA_ROOT}')
        break

RUN_ROOT = Path('/kaggle/working/roboat')
SCRIPT = Path('/kaggle/working/train_run_c.py')

SCRIPT_B64 = "..."

SCRIPT.write_bytes(base64.b64decode(SCRIPT_B64))
print('Wrote', SCRIPT, '| size:', SCRIPT.stat().st_size, 'bytes')
print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT: ', RUN_ROOT)


## 1. Data and Augmentation Audit
Cached report reuses in 0.0s.


In [ ]:
cached_audit = RUN_ROOT / 'audit_report.json'
if cached_audit.is_file():
    print(f'Reusing cached audit report: {cached_audit}')
else:
    subprocess.run([sys.executable, str(SCRIPT), '--mode', 'audit', '--data-root', str(DATA_ROOT), '--run-root', str(RUN_ROOT)], check=True)


In [ ]:
from IPython.display import display, Image
preview = RUN_ROOT / 'augmentation_preview.jpg'
if preview.is_file():
    display(Image(filename=str(preview)))


## 2. Stage-2 Control Arm Training & Dual-Protocol Evaluation
Resumes from Epoch 44 checkpoint (`last.pt`), trains to 100 epochs, and automatically executes Dual-Protocol (Standard iou=0.60 + TTA iou=0.55) stratified test evaluation.


In [ ]:
# Auto-detect previous run checkpoint from dataset_sources or input
CONTROL_RESUME_FROM = None
candidates = [
    Path('/kaggle/input/roboat-run-c-v4-output/run_c_control'),
    Path('/kaggle/input/roboat-run-c-v4-output/roboat/run_c_control'),
    Path('/kaggle/input/roboat-run-c-v2-output/run_c_control'),
    Path('/kaggle/input/roboat-run-c-v2-output/roboat/run_c_control'),
    Path('/kaggle/input/roboat-run-c-training/roboat/run_c_control'),
    Path('/kaggle/input/roboat-run-c-training/run_c_control'),
]
input_dir = Path('/kaggle/input')
if input_dir.is_dir():
    for match in input_dir.rglob('run_c_control'):
        if match.is_dir() and match not in candidates:
            candidates.append(match)

for p in candidates:
    if (p / 'weights' / 'last.pt').is_file() or (p / 'weights' / 'resume_safe.pt').is_file() or (p / 'weights' / 'best.pt').is_file() or (p / 'last.pt').is_file():
        CONTROL_RESUME_FROM = p
        print(f'[resume] Auto-detected previous run checkpoint: {CONTROL_RESUME_FROM}')
        break

cmd = [
    sys.executable, str(SCRIPT),
    '--mode', 'train',
    '--arm', 'control',
    '--data-root', str(DATA_ROOT),
    '--run-root', str(RUN_ROOT),
    '--total-epochs', '100',
    '--lr0', '0.003',
    '--close-mosaic', '15',
    '--scale', '0.25',
    '--mosaic', '0.80',
    '--batch', '32',
    '--workers', '4',
    '--max-hours', '10.2',
]
if CONTROL_RESUME_FROM is not None:
    cmd += ['--resume', '--resume-from', str(CONTROL_RESUME_FROM)]
else:
    print('[train] Starting fresh control arm training')

print('Executing:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## 3. Results Trajectory & Evaluation Report


In [ ]:
import json
from IPython.display import display, Image

eval_file = RUN_ROOT / 'eval_results_control.json'
if eval_file.is_file():
    print(f'Evaluation report: {eval_file}')
    data = json.loads(eval_file.read_text(encoding='utf-8'))
    print(json.dumps(data.get('summary', {}), indent=2))

for img_name in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    p = RUN_ROOT / 'run_c_control' / img_name
    if p.is_file():
        print(f'Displaying: {img_name}')
        display(Image(filename=str(p)))
